# b3_retro

> Phase 22 (`specs/multiagent/phase22/`, experiment B3): the retrospective check on paper2's calls (FR-D2, H22-2, P22-11). Per fold (`S1`, `S4`, `S4b`, and the secondary `P14`): θ0 fit without the fold's cells, refit on that study's own pilot (seed 1890; Phase 14's at seed 900), each study's frozen estimate rule, and the scores of the five lines against the saved Haiku results; for every behaviour-1 candidate, since the knob was chosen on the same folds (P22-18). `retroCellFn` plays the retrospective items' numpy cells, `pilotTargets` maps a study's pilot to refit targets.

In [ ]:
#| default_exp b3_retro

In [ ]:
#| hide
from nbdev.showdoc import *

In [ ]:
#| export
import numpy as np

from explRL.b3_cells import CANDIDATES, FIT_CELLS, FOLDS, SIM_SEEDS, _fc, claudeRows, claudeTargets, grid, parseKey, thetaKey
from explRL.b3_effects import RETRO_EFFECTS, RETRO_STUDIES, p14Estimate, retroSaved
from explRL.b3_fit import gridSim, loss, pilotScale, refit, simRow
from explRL.b3_forecast import call, estimate, forecastNaive, forecastPilot, forecastSim

RETRO_PILOTS = dict(S1='colony/pilot', S4='colony/stage4pilot', S4b='colony/stage4bpilot', P14='scale/pilot')
_PRE4 = ('colony/stage1', 'colony/stage32', 'colony/stage3', 'colony/stage64')
RETRO_PRIOR = dict(S1=(), S4=_PRE4, S4b=_PRE4 + ('colony/stage4',), P14=())   # Claude cells run before each study's pilot
CONCLUSIVE = ('S1', 'S4', 'S4b')                                               # paper2's 11 conclusive calls are in these folds
FAULT_CELLS = ([_fc('colony/stage1', 'colony1', a, 'error', N) for a in ('plain', 'colony') for N in (8, 16)] +   # not fit (P22-3)
               [_fc('colony/stage32', 'colony32', a, f, 32) for a in ('plain', 'colonyGuards') for f in ('error', 'collude')])


def faultChecks(theta, workers=8, seeds=SIM_SEEDS, out=None, root=None):
    "P22-3: the error and collude cells as held-out checks: contamination, Claude against θ_default and θ (the fit on all cells)."
    sims, T = gridSim(FAULT_CELLS, [{}, theta], workers, seeds, out), claudeTargets(claudeRows(root), FAULT_CELLS)
    k = thetaKey(theta)
    return dict(theta=theta, cells={c: dict(claude=T[c][0], default=sims['default'][c], fitted=sims[k][c]) for c in T},
                loss=dict(default=loss({}, list(T), sims, T), fitted=loss(theta, list(T), sims, T)))


def _cell(arm, fault, N, tag='retro'):
    return dict(id=f'{tag}:{arm}|{fault}:{N}', kind='p14' if arm == 'baseline' else 'colony', arm=arm, fault=fault, N=int(N),
                target='pooledShare' if fault == 'clean' else 'contamination')


def retroCellFn(arm, fault, N, seed, theta):
    "One numpy board of a retrospective item's cell (Phase 14's plain board for arm 'baseline'), with world['b3'] = θ."
    return simRow(_cell(arm, fault, N), seed, theta)


def pilotTargets(fold, pilotRows, rows):
    """A study's pilot as refit targets: clean boards on pooledShare, liar boards on contamination (rows without the measure are
    skipped); s from the Claude fit cells outside the fold (pilotScale, one board each) -> ([cell], [target], refit params)."""
    keep = [c for c in FIT_CELLS if c['id'] not in FOLDS[fold]]
    cells, tg = [], []
    for r in pilotRows:
        arm, fault = (r['level'].split('|') + ['clean'])[:2]
        c = _cell(arm, fault, float(r['N']), 'pilot')
        if r.get(c['target']) not in (None, '') and np.isfinite(float(r[c['target']])):
            cells.append(c)
            tg.append(dict(cell=c['id'], measure=c['target'], value=float(r[c['target']]), s=pilotScale(rows, c['target'], 1, keep)))
    params = tuple(p for p, m in (('trustOne', 'contamination'), (None, 'pooledShare')) if any(t['measure'] == m for t in tg))
    return cells, tg, params

In [ ]:
#| export
def retroFold(fold, knob, theta0, tau, thetas, pilotRows, priorRows, rows, saved, workers=8, seeds=SIM_SEEDS, out=None):
    """One fold for one candidate knob: refit θ0 on the fold's pilot over `thetas` (the candidate's grid), then the five lines on
    the fold's items and their scores against the saved Haiku intervals."""
    effs = [e for e in RETRO_EFFECTS if e.id in RETRO_STUDIES[fold]['ids']]
    cells, tg, params = pilotTargets(fold, pilotRows, rows)
    params = tuple(knob if p is None else p for p in params)
    fixed = [k for k in ('trustOne', knob) if k not in params]
    cand = [th for th in thetas if all(th.get(k, 0.0) == theta0.get(k, 0.0) for k in fixed)]
    thp = refit(theta0, tau, tg, gridSim(cells, cand, workers, seeds, out), params) if tg and params else dict(theta0)
    est = p14Estimate if fold == 'P14' else estimate
    th = dict(fitted=thp, priorOnly=theta0, unfitted={})
    lines = {k: forecastSim(effs, retroCellFn, t, seeds, est) for k, t in th.items()}
    if fold != 'P14':                                 # team size is not a design: no pilot-only or every-design-helps (P22-11)
        lines.update(pilotOnly=forecastPilot(effs, pilotRows, priorRows, est), naive=forecastNaive(effs))
    items = {e.id: dict(saved[e.id], lines={k: v[e.id]['sign'] for k, v in lines.items() if e.id in v},
                        calls={k: call(v[e.id]['sign'], saved[e.id]['ci']) for k, v in lines.items() if e.id in v})
             for e in effs if e.id in saved}
    for i in items.values():
        i['unfittedMatches'] = i['lines']['unfitted'] == i['unfitted']
    return dict(fold=fold, knob=knob, theta=dict(th, fitted=thp), pilot=tg, params=list(params), items=items)


def _counts(items, lines=('fitted', 'priorOnly', 'unfitted', 'pilotOnly', 'naive')):
    return {k: {o: sum(i['calls'].get(k) == o for i in items) for o in ('right', 'wrong', 'no call')} for k in lines}


def retrospective(fitRes, workers=8, seeds=SIM_SEEDS, root=None, thetas=None, folds=('S1', 'S4', 'S4b', 'P14'), knobs=CANDIDATES,
                  cache=None):
    """FR-D2: every fold for every candidate: θ0 = the candidate's held-out fit (the winner's fine-grid fit), τ its bootTau without the fold's cells; the
    winner refits on its fine grid, the others on the coarse grid (`thetas(knob)` overrides). Counts on the 11 conclusive calls
    (H22-2: fitted right > the unfitted line's, 9 in paper2), P14 apart, whether the unfitted line reproduces the frozen signs, and
    the error and collude faults as held-out checks of the fit on all cells (P22-3)."""
    from explRL.b3_cells import claudeRows
    from explRL.bench_grid import readCSV
    from explRL.train import demo_artifact_dir
    R = demo_artifact_dir(None, 'multiagent') if root is None else root
    rows, saved, out = claudeRows(R), retroSaved(R.parents[1]), {}
    for k in knobs:
        g = thetas(k) if thetas else grid(('trustOne', k), 'fine' if k == fitRes['knob'] else None)
        out[k] = {}
        for f in folds:
            held = fitRes['heldOut']['fine' if k == fitRes['knob'] else 'coarse']
            th0 = (held[f] if k == fitRes['knob'] else held[k][f])['theta']
            prior = [r for d in RETRO_PRIOR[f] for r in readCSV(R / d / 'runs.csv') if r.get('runner', 'llm') == 'llm']
            out[k][f] = retroFold(f, k, th0, fitRes['tauBy'][k][f], g, readCSV(R / RETRO_PILOTS[f] / 'runs.csv'), prior, rows,
                                  saved, workers, seeds, cache)
    summary = {}
    for k in out:
        concl = [i for f in CONCLUSIVE if f in out[k] for i in out[k][f]['items'].values() if i['calls']['unfitted'] != 'no call']
        c = _counts(concl)
        summary[k] = dict(conclusive=len(concl), counts=c, H22_2=c['fitted']['right'] > c['unfitted']['right'],
                          P14=_counts(out[k]['P14']['items'].values(), ('fitted', 'priorOnly', 'unfitted')) if 'P14' in out[k] else None,
                          unfittedMismatch=[i for f in out[k] for i, v in out[k][f]['items'].items() if not v['unfittedMatches']])
    return dict(knob=fitRes['knob'], folds=out, summary=summary, faults=faultChecks(fitRes['fits']['all'], workers, seeds, cache, R),
                caveat='in-sample: the behaviours were named after these misses, and the knob was chosen on the same folds (P22-18)')

## Tests

In [ ]:
# test_b3_retro (FR-D2, H22-2, P22-11): one fold on a tiny grid and 3 seeds: the pilot targets, the refit params, the five
# lines and their calls, the unfitted check, and the summary's shape. Needs the saved Claude rows and pilots (results/ is
# gitignored: CI and the docs build skip it)
import tempfile
from pathlib import Path
from explRL.b3_cells import claudeRows
from explRL.bench_grid import readCSV
from explRL.train import demo_artifact_dir

R = demo_artifact_dir(None, 'multiagent')
if not all((R / f / 'runs.csv').exists() for f in {c['folder'] for c in FIT_CELLS} | {RETRO_PILOTS['S1'], RETRO_PILOTS['S4']}):
    print('skipped: the saved Claude rows and pilots are not here (results/ is gitignored)')
else:
    rows = claudeRows(R)
    cells, tg, params = pilotTargets('S1', readCSV(R / 'colony/pilot/runs.csv'), rows)
    assert len(cells) == 4 and params == ('trustOne', None) and {t['measure'] for t in tg} == {'pooledShare', 'contamination'}
    assert all(t['s'] > 0 for t in tg)
    cells, tg, params = pilotTargets('S4', readCSV(R / 'colony/stage4pilot/runs.csv'), rows)
    assert len(cells) == 3 and params == ('trustOne',)
    fake = dict(knob='spread', tauBy=dict(spread=dict(trustOne=0.1, spread=0.1)),
                heldOut=dict(fine={f: dict(theta=dict(trustOne=0.5)) for f in FOLDS}, coarse={}))
    tiny = lambda k: [dict(trustOne=a, spread=b) for a in (0.0, 0.5, 1.0) for b in (0.0, 0.5)]
    fake['fits'] = dict(all=dict(trustOne=0.5))
    fake['tauBy'] = dict(spread={f: dict(trustOne=0.1, spread=0.1) for f in FOLDS})
    r = retrospective(fake, workers=0, seeds=range(3000, 3003), thetas=tiny, folds=('S1',), knobs=('spread',), cache=tempfile.mkdtemp())
    S = r['folds']['spread']['S1']
    assert set(S['items']) == {'H18_1', 'H18_2', 'H18_3', 'H18_4', 'H18_6', 'colonyVsConfirm'} and S['params'] == ['trustOne', 'spread']
    assert all(set(i['calls']) == {'fitted', 'priorOnly', 'unfitted', 'pilotOnly', 'naive'} for i in S['items'].values())
    assert S['theta']['unfitted'] == {} and S['theta']['priorOnly'] == dict(trustOne=0.5)
    sm = r['summary']['spread']
    assert sm['conclusive'] == 4 and sm['counts']['naive']['right'] == 3 and isinstance(sm['H22_2'], bool)
    assert S['items']['H18_3']['lines']['pilotOnly'] == '-'                     # computed from the 16-agent pilot, not a fallback
    F = r['faults']
    assert len(F['cells']) == 8 and all(set(v) == {'claude', 'default', 'fitted'} for v in F['cells'].values()) and F['loss']['default'] > 0

In [ ]:
#| hide
import nbdev; nbdev.nbdev_export()